# 离线多端口共享原型

本示例只生成合成数据，不连接硬件。演示公开别名、提供者内的子进程、按需订阅以及稳定样本身份。需要 Linux、RSim 和可用的 DDS；普通消费者无需 ROS。

In [ ]:
import asyncio
from pathlib import Path
import os
import uuid
import numpy as np
from rsim import (Component, Runtime, SharedProvider, SharedComponent, PortSpec,
                  ProcessPlacement, serve_shared, describe_shared,
                  ProviderDisconnected, PortNotBound)

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'pyproject.toml').is_file() and (p / 'rsim').is_dir())
assets = root / 'assets' / 'shared-ports'
assets.mkdir(parents=True, exist_ok=True)


In [ ]:
class ToyMapping(Component):
    def __init__(self):
        super().__init__()
        self.pose = self.signal('pose', clock='demo')
        self.map = self.signal('map', history=2, clock='demo')
        self.revision = dict(session_id=uuid.uuid4().hex, revision=1)

    async def open(self):
        points = np.arange(300, dtype=np.float32).reshape(-1, 3)
        points.flags.writeable = False
        await self.map.publish(points, stamp_ns=100, clock='demo', metadata=self.revision)
        self.task('pose', self.tick, hz=20)

    async def tick(self):
        await self.pose.publish(dict(x=0.0, y=0.0, pid=os.getpid()),
                                stamp_ns=100, clock='demo', metadata=self.revision)

source = ToyMapping()
assembly = Component()
assembly.expose('pose', source.pose)
assembly.expose('map', source.map)
key = 'example:' + uuid.uuid4().hex
provider = SharedProvider(assembly, key=key, placement={source: ProcessPlacement('mapping')})
provider_task = asyncio.create_task(serve_shared(provider))
async with asyncio.timeout(15):
    while True:
        if provider_task.done():
            await provider_task
        try:
            manifest = await describe_shared(key)
            break
        except ProviderDisconnected:
            await asyncio.sleep(.05)
print(manifest['interface_version'], sorted(manifest['ports']))


In [ ]:
mapper = SharedComponent(key=key, ports={
    'pose': PortSpec(clock='demo'), 'map': PortSpec(clock='demo', history_capacity=2),
})
async with Runtime(mapper.pose):
    pose = await mapper.pose.get(timeout=10)
    print(pose.data, pose.sample_id)
    try:
        await mapper.map.get()
    except PortNotBound:
        print('map 未被订阅；需要时将其加入 Runtime 根。')

async with Runtime(mapper.pose, mapper.map):
    frame = await mapper.map.get(timeout=10)
    identity = frame.sample_id
    assert isinstance(frame.data, np.memmap) and not frame.data.flags.writeable
    np.save(assets / 'synthetic-map.npy', frame.data)
    print(frame.metadata, frame.data.shape)

# 重建通道，但原来的静态地图样本仍是同一次发布。
async with Runtime(mapper.map):
    assert (await mapper.map.get(timeout=10)).sample_id == identity


In [ ]:
provider_task.cancel()
await asyncio.gather(provider_task, return_exceptions=True)
# 已持有的只读映射在 provider 退出后仍然可读。
assert np.array_equal(frame.data, np.arange(300, dtype=np.float32).reshape(-1, 3))
